# Caso de Estudio 3: Aprendizaje Semi-Supervisado
Este notebook cubre las técnicas de **Aprendizaje Semi-Supervisado**:
1. **Label Propagation** (Propagación de etiquetas vía K-Means representantes).
2. **Self-Training Classifier** (Auto-etiquetado supervisado iterativo).


In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings('ignore')


## 1. Carga y Ocultamiento de Etiquetas


In [2]:
X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

# Simulación: 95% de datos de entrenamiento se quedan sin etiquetar (representados con -1 en sklearn)
y_train_semi = np.copy(y_train)
random_unlabeled_points = np.random.rand(len(y_train)) < 0.95
y_train_semi[random_unlabeled_points] = -1

# Conteo de datos etiquetados reales
print("Ejemplos etiquetados reales en Train:", np.sum(y_train_semi != -1))


Ejemplos etiquetados reales en Train: 46


## 2. Self-Training Classifier


In [3]:
# Inicializamos un clasificador base
base_model = LogisticRegression(max_iter=5000, random_state=42)

# Self-Training usando un umbral de confianza del 80% para propagar etiquetas automáticas
self_training = SelfTrainingClassifier(base_model, threshold=0.80)
self_training.fit(X_train, y_train_semi)

y_pred = self_training.predict(X_test)
print(f"Accuracy de Self-Training en Test: {accuracy_score(y_test, y_pred)*100:.2f}%")


Accuracy de Self-Training en Test: 87.78%


## 3. Banco de Preguntas Teóricas (Nivel Examen)
*   **Pregunta 1:** ¿Cómo decide el `SelfTrainingClassifier` qué muestras no etiquetadas incluir en el siguiente ciclo de entrenamiento?
    *   *Respuesta:* Utiliza la probabilidad predictiva (`predict_proba`) de la clase mayoritaria de cada muestra no etiquetada. Si esta probabilidad supera el umbral configurado (`threshold=0.80`), la predicción se asume como su etiqueta verdadera y se añade al set etiquetado para el siguiente re-entrenamiento.
*   **Pregunta 2:** ¿Cuál es el peligro de configurar un `threshold` demasiado bajo (ej: 0.51) en Self-Training?
    *   *Respuesta:* El modelo incorporará etiquetas con muy bajo nivel de certeza. Si el modelo comete un error y pseudo-etiqueta mal una muestra, en la siguiente iteración se entrenará con sus propios errores, desencadenando un bucle de degradación del modelo.
